# Multi-Regional AeroMAPS Scenarios

This notebook demonstrates the multi-regional feature in AeroMAPS using the `MultiRegionalProcess` class.

## Overview

The multi-regional mode allows you to:
1. Run multiple regional AeroMAPS scenarios
2. Choose between two execution modes:
   - **unified_mda**: All regions in a single MDAChain (memory-intensive, handles coupling)
   - **separate_processes**: Each region executed independently (scalable, parallel-ready)
3. Aggregate regional outputs into global metrics using configurable rules
4. Access full regional process API (plotting, data export, etc.)

## Configuration

Multi-regional mode is configured via a YAML file with a `regionalisation` section containing:
- `execution_mode`: "unified_mda" or "separate_processes" (default)
- `regions`: mapping of region IDs to their config files
- `aggregation`: rules for computing global metrics

## 1. Prepare Regional Data

First, we generate the partitioning data for each region.

In [ ]:
from aeromaps.utils.functions import create_partitioning

# Generate partitioning data for both regions
create_partitioning(file="data/region_france/aeroscope_data.csv", path="data/region_france")
create_partitioning(file="data/region_germany/aeroscope_data.csv", path="data/region_germany")
print("✓ Partitioning data generated for both regions")

## 2. Create Multi-Regional Process

Use `create_multi_regional_process()` for explicit multi-regional creation, or `create_process()` which auto-detects from the config file.

The `execution_mode` in `regionalisation.yaml` controls how regions are executed:
- `"separate_processes"` (default): More scalable, allows parallel execution
- `"unified_mda"`: All disciplines in one GEMSEO MDAChain

In [ ]:
%matplotlib widget
from aeromaps import create_process, create_multi_regional_process

# Option 1: Auto-detect from config (returns MultiRegionalProcess if regionalisation section exists)
# process = create_process(configuration_file="regionalisation.yaml")

# Option 2: Explicit multi-regional creation (recommended for clarity)
process = create_multi_regional_process(configuration_file="regionalisation.yaml", disable_execution_statistics=True)

print(f"✓ Multi-regional process created")
print(f"  Execution mode: {process.execution_mode}")
print(f"  Regions: {process.list_regions()}")
print(f"  Regional processes available: {list(process.regional_processes.keys())}")

## 3. Execute Multi-Regional Analysis

The `compute()` method runs all regional scenarios. In `separate_processes` mode, you can enable parallel execution with `parallel=True`.

In [ ]:
import time

# Execute the multi-regional analysis
# In 'separate_processes' mode, you can use parallel=True for concurrent execution
start_time = time.time()
process.compute(parallel=True, max_workers=16)  # Set parallel=True for parallel execution (separate_processes mode only)
elapsed_time = time.time() - start_time
print(f"✓ Computation complete in {elapsed_time:.2f} seconds")

## 4. Access Regional Outputs

You can access outputs for specific regions or all regions using the dedicated methods.

In [ ]:
import pandas as pd

# Method 1: Get aggregated outputs from the multi-regional process
fr_outputs = process.get_regional_outputs("FR")
de_outputs = process.get_regional_outputs("DE")
global_outputs = process.get_global_outputs()

# Method 2: Access regional processes directly (full AeroMAPSProcess API)
fr_process = process.regional_processes["FR"]
de_process = process.regional_processes["DE"]

print("Regional process access:")
print(f"  FR process has {len(fr_process.disciplines)} disciplines")
print(f"  FR parameters: end_year = {fr_process.parameters.end_year}")

print("\nAvailable global aggregated variables:")
print(f"  {list(global_outputs.keys())}")

## 5. Compare Regional and Global Results

Let's compare CO2 emissions across regions and verify aggregation.

In [ ]:
import numpy as np

years = [2020, 2030, 2040, 2050]

print("=" * 60)
print("MULTI-REGIONAL RESULTS")
print("=" * 60)

# CO2 Emissions comparison
print("\nCO2 EMISSIONS (Mt):")
print("-" * 40)

data = {"Year": years}

if "co2_emissions" in fr_outputs:
    data["France"] = [fr_outputs["co2_emissions"][y] / 1e9 for y in years]
if "co2_emissions" in de_outputs:
    data["Germany"] = [de_outputs["co2_emissions"][y] / 1e9 for y in years]
if "co2_emissions" in global_outputs:
    data["GLOBAL"] = [global_outputs["co2_emissions"][y] / 1e9 for y in years]

df = pd.DataFrame(data).set_index("Year")
display(df)

In [ ]:
# Verify aggregation correctness
print("\nVERIFICATION - CO2 emissions 2050:")
print("-" * 40)

fr_val = fr_outputs.get("co2_emissions_including_energy", pd.Series())[2050] if "co2_emissions_including_energy" in fr_outputs else 0
de_val = de_outputs.get("co2_emissions_including_energy", pd.Series())[2050] if "co2_emissions_including_energy" in de_outputs else 0
global_val = global_outputs.get("co2_emissions_including_energy", pd.Series())[2050] if "co2_emissions_including_energy" in global_outputs else 0

print(f"  FR + DE = {(fr_val + de_val):.2f} Mt")
print(f"  Global  = {global_val:.2f} Mt")
print(f"  Match: {np.isclose(fr_val + de_val, global_val)}")

## 6. Access Regional Process Data Directly

In `separate_processes` mode, each regional process has its own MDAChain that can be accessed directly.

In [ ]:
# Access regional process MDAChain data directly
fr_process = process.regional_processes["FR"]
fr_local_data = fr_process.mda_chain.local_data

print(f"FR process local_data has {len(fr_local_data)} variables")
print(f"\nSample variables from FR:")
for key in sorted(fr_local_data.keys())[:10]:
    print(f"  {key}")

# You can also use the standard AeroMAPSProcess methods
# fr_process.plot("co2_emissions")  # Plot for France
# fr_process.write_excel("france_results.xlsx")  # Export to Excel

## 7. Summary

The multi-regional feature provides:

1. **Two Execution Modes**:
   - `separate_processes`: Each region runs independently (scalable, parallel-ready)
   - `unified_mda`: All disciplines in one MDAChain (for inter-regional coupling)

2. **Full Regional Process Access**: Access each region's `AeroMAPSProcess` for plotting, export, etc.

3. **Configurable Aggregation**: Define sum or weighted average rules in YAML

4. **Simple API**: Use `create_multi_regional_process()` or auto-detect with `create_process()`

### Configuration Example

```yaml
# regionalisation.yaml
regionalisation:
  execution_mode: "separate_processes"  # or "unified_mda"
  global_namespace: "overall"
  
  regions:
    FR:
      config_file: "data/region_france/config.yaml"
    DE:
      config_file: "data/region_germany/config.yaml"
  
  aggregation:
    sum:
      - co2_emissions
      - rpk
    weighted_average:
      - variable: load_factor
        weight_by: rpk
```

### Usage

```python
from aeromaps import create_multi_regional_process

process = create_multi_regional_process("regionalisation.yaml")
process.compute(parallel=True, max_workers=4)  # Parallel execution

# Access regional data
fr_process = process.regional_processes["FR"]
fr_process.plot("co2_emissions")

# Access global aggregates
global_outputs = process.get_global_outputs()
```

In [ ]:
# Cleanup
from aeromaps.utils.functions import clean_notebooks_on_tests

clean_notebooks_on_tests(globals(), force_cleanup=True)